
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# LAB- Evaluation with Mosaic AI Agent Evaluation

In this lab, you will have the opportunity to evaluate a RAG chain model **using Mosaic AI Agent Evaluation Framework.**

**Lab Outline:**

*In this lab, you will complete the following tasks:*

- **Task 1**: Define a custom Gen AI evaluation metric.

- **Task 2**: Conduct an evaluation test using the Agent Evaluation Framework.

- **Task 3**: Analyze the evaluation results through the user interface.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**



## Classroom Setup

Install required libraries.

In [0]:
%pip install -U -qq databricks-agents databricks-sdk databricks-vectorsearch databricks-langchain langchain==0.3.7 langchain-community==0.3.7 mlflow>=3.0 databricks-feature-engineering --upgrade

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.12.0 which is incompatible.
msal 1.29.0 requires cryptography<45,>=2.5, but you have cryptography 46.0.3 which is incompatible.
oci 2.126.4 requires cryptography<43.0.0,>=3.2.1, but you have cryptography 46.0.3 which is incompatible.
pyopenssl 23.2.0 requires cryptography!=40.0.0,!=40.0.1,<42,>=38.0.0, but you have cryptography 46.0.3 which is incompatible.
tensorboard-plugin-profile 2.15.1 requires protobuf<5.0.0dev,>=3.19.6, but you have protobuf 5.29.5 which is incompatible.
tensorflow 2.16.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.5 which is incompatible.
ydata-profiling 4.5.1 requires pydantic<2,>=1.8.1, but you have pydantic 2.12.5 which is incompatible.
Note: you

In [0]:
dbutils.library.restartPython()

Before starting the lab, run the provided classroom setup script. This script will define configuration variables necessary for the lab. Execute the following cell:

In [0]:
%run ../Includes/Classroom-Setup-04


The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


Created eval_dataset with 12 evaluation items


**Other Conventions:**

Throughout this lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

Username:          labuser12546244_1764715360@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12546244_1764715360
Working Directory: /Volumes/dbacademy/ops/labuser12546244_1764715360@vocareum_com
Dataset Location:  NestedNamespace (news='/Volumes/dbacademy_news/v01', arxiv='/Volumes/dbacademy_arxiv/v01')


## Lab Overview




## Evaluation Dataset

In this lab, you will work with the same dataset utilized in the demos. This dataset contains sample queries along with their corresponding expected responses, which are generated using synthetic data.

In [0]:
display(DA.eval_df)

request,expected_response,evolution_type,episode_done
"What are the limitations of symbolic planning in task and motion planning, and how can leveraging large language models help overcome these limitations?","Symbolic planning in task and motion planning can be limited by the need for explicit primitives and constraints. Leveraging large language models can help overcome these limitations by enabling the robot to use language models for planning and execution, and by providing a way to extract and leverage knowledge from large language models to solve temporally extended tasks.",simple,true
"What are some techniques used to fine-tune transformer models for personalized code generation, and how effective are they in improving prediction accuracy and preventing runtime errors?","The techniques used to fine-tune transformer models for personalized code generation include ﬁne-tuning transformer models, adopting a novel approach called Target Similarity Tuning (TST) to retrieve a small set of examples from a training bank, and utilizing these examples to train a pretrained language model. The effectiveness of these techniques is shown in the improvement in prediction accuracy and the prevention of runtime errors.",simple,true
How does the PPO-ptx model mitigate performance regressions in the few-shot setting?,"The PPO-ptx model mitigates performance regressions in the few-shot setting by incorporating pre-training and fine-tuning on the downstream task. This approach allows the model to learn generalizable features and adapt to new tasks more effectively, leading to improved few-shot performance.",simple,true
How can complex questions be decomposed using successive prompting?,"Successive prompting is a method for decomposing complex questions into simpler sub-questions, allowing language models to answer them more accurately. This approach was proposed by Dheeru Dua, Shivanshu Gupta, Sameer Singh, and Matt Gardner in their paper 'Successive Prompting for Decomposing Complex Questions', presented at EMNLP 2022.",simple,true
"Which entity type in Named Entity Recognition is likely to be involved in information extraction, question answering, semantic parsing, and machine translation?",Organization,reasoning,true
What is the purpose of ROUGE (Recall-Oriented Understudy for Gisting Evaluation) in automatic evaluation methods?,"ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is used in automatic evaluation methods to evaluate the quality of machine translation. It calculates N-gram co-occurrence statistics, which are used to assess the similarity between the candidate text and the reference text. ROUGE is based on recall, whereas BLEU is based on accuracy.",simple,true
"What are the challenges associated with Foundation SSL in CV, and how do they relate to the lack of theoretical foundation, semantic understanding, and explicable exploration?","The challenges associated with Foundation SSL in CV include the lack of a profound theory to support all kinds of tentative experiments, and further exploration has no handbook. The pretrained LM may not learn the meaning of the language, relying on corpus learning instead. The models cannot reach a better level of stability and match different downstream tasks, and the primary method is to increase data, improve computation power, and design training procedures to achieve better results. The lack of theoretical foundation, semantic understanding, and explicable exploration are the main challenges in Foundation SSL in CV.",simple,true
How does ChatGPT handle factual input compared to GPT-3.5?,"ChatGPT handles factual input better than GPT-3.5, with a 21.9% increase in accuracy when the premise entails the hypothesis. This is possibly related to the preference for human feedback in ChatGPT's RLHF design during model training.",simple,true
What are some of the challenges in understanding natural language commands for robotic navigation and mobile manipulation?,"Some challenges in underst

## Load the Model

A RAG chain has been created and registered for use in this lab. The model details are provided below.

**📌 Note:** If you are using your own workspace to run this lab, you must manually execute **`00 - Build Model / 00-Build Model`**.

In [0]:
import mlflow

catalog_name = "genai_shared_catalog_03"
schema_name = f"ws_{spark.conf.get('spark.databricks.clusterUsageTags.clusterOwnerOrgId')}"

mlflow.set_registry_uri("databricks-uc")

model_uri = f"models:/{catalog_name}.{schema_name}.rag_app/1"
model_name = f"{catalog_name}.{schema_name}.rag_app"

## Task 1 - Define A Custom Metric

For this task, define a custom metric to evaluate whether the generated **"ANSWER"** from the RAG chain is easily readable by a non-expert user.

In [0]:
from mlflow.metrics.genai import make_genai_metric_from_prompt

## Prompt for LLM as judge to determine if the generated response is easily readable by non-academic or expert readers
eval_prompt = "Your task is to determine whether the generated response easily readable by non-academic or expert readers. This was the content: '{retrieved_context}'"

## Use Llama-3 as LLM
llm="endpoints:/databricks-meta-llama-3-3-70b-instruct"

## Define the metric
is_readable = make_genai_metric_from_prompt(
    name="is_readable",
    judge_prompt=eval_prompt,
    model=llm,
    metric_metadata={"assessment_type": "ANSWER"},
)

/root/.ipykernel/1444/command-8080809395929266-3071364902:10: FutureWarning: ``mlflow.metrics.genai.genai_metric.make_genai_metric_from_prompt`` is deprecated since 3.4.0. Use the new GenAI evaluation functionality instead. See https://mlflow.org/docs/latest/genai/eval-monitor/legacy-llm-evaluation/ for the migration guide.
  is_readable = make_genai_metric_from_prompt(


##Task 2 - Run Evaluation Test

Next, run an evaluation using the custom metric you defined. Ensure that you select **Mosaic AI Agent Evaluation** as the evaluation type.


In [0]:
with mlflow.start_run(run_name="lab_04_agent_evaluation"):
    eval_results = mlflow.evaluate(
        data = DA.eval_df,
        model = model_uri,
        model_type = "databricks-agent",
        extra_metrics=[is_readable]
    )

/local_disk0/.ephemeral_nfs/envs/pythonEnv-76ff50d1-b874-4c9e-9890-1b88a29adb41/lib/python3.11/site-packages/mlflow/models/evaluation/deprecated.py:9: FutureWarning: The `mlflow.evaluate` API has been deprecated as of MLflow 3.0.0. Please use these new alternatives:

 - For traditional ML or deep learning models: Use `mlflow.models.evaluate`, which maintains full compatibility with the original `mlflow.evaluate` API.

 - For LLMs or GenAI applications: Use the new `mlflow.genai.evaluate` API, which offers enhanced features specifically designed for evaluating LLMs and GenAI applications.

  warnings.warn(


2025/12/02 22:51:49 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - databricks-vectorsearch (current: 0.63, required: databricks-vectorsearch==0.60)
 - mlflow (current: 3.6.0, required: mlflow==3.4.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/12/02 22:51:49 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-9cdcb65cf6644e938014daff0d62cabc
2025/12/02 22:51:49 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/12/02 22:51:50 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-9cdcb65cf6644e938014daff0d62cabc
2025/12/02 22:51:50 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if need

Evaluating:   0%|          | 0/12 [Elapsed: 00:00, Remaining: ?] 

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

/local_disk0/.ephemeral_nfs/envs/pythonEnv-76ff50d1-b874-4c9e-9890-1b88a29adb41/lib/python3.11/site-packages/databricks/rag_eval/mlflow/databricks_rag_evaluator.py:158: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  search_traces_df = mlflow.search_traces(experiment_ids=[experiment_id], run_id=run_id)


<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

[Trace(trace_id=tr-b03400495dd39106338aa6e74a9948e4), Trace(trace_id=tr-43b36e99bedf5e09621dfe671e2693bf), Trace(trace_id=tr-b8be077e6ff4e2e08de4aef42d08c3cb), Trace(trace_id=tr-4ea9f18f8ac2e8953cf304bec1e4a1b8), Trace(trace_id=tr-52fb67c0d631534725bb21679643b3cb), Trace(trace_id=tr-5e3d478646a10feeaaa5fe68164095f2), Trace(trace_id=tr-c9ba85b81c3b90a88d64035dd5efb3f6), Trace(trace_id=tr-d07e9f6e5b15cb8e34f8c77b195257d1), Trace(trace_id=tr-15f128883374e5acbe61a394fbdcb094), Trace(trace_id=tr-2fa445b107c17d2b15afae285a82d3e4)]

## Task 3 - Review Evaluation Results

Review the evaluation results in the **Experiments** section. Examine the following information regarding this evaluation:

- Token usage

- Model metrics

- Results of the custom metric defined earlier ("readability")

## Conclusion

In this lab, you evaluated a RAG chain using the Mosaic AI Evaluation Framework library. You began by loading the dataset and RAG model. Then, you defined a custom metric and initiated the evaluation process. Finally, you reviewed the results through the user interface.

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>